## Eksperiment 3 eeltöötlus

Erinevus eksperiment kahest:
* Juurde on lisatud ükskilausete korpus (tehtud sarnane eeltöötlus kui otsekõne ja jutustaja korpustele)
* Varasemalt mitte kasutatud testhulgast on lauseid validatsioonihulka juurde lisatud, et see oleks stabiilsem.



0_1: Resample 22050 : resample_folder.sh

0_2: Vaikuse eemaldamine, et oleks sarnasem jutustaja ja otsekõne korpustega (kogu vaikus algusest, 0,5 sekundit lõpust): trim_yksiklaused.sh

1. Üksiklausete tekstide ja wav-failide eraldamine kaustadesse

In [1]:
import os
import shutil
import re
from sklearn.model_selection import train_test_split

In [ ]:
sentences_1 = "/home/kiisu/Documents/matcha/data/peeter_yksiklaused/828lauset _ptr.kj1"
sentences_2 = "/home/kiisu/Documents/matcha/data/peeter_yksiklaused/1020lauset_ptr.kj1"

yksiklaused_txt_path = "/home/kiisu/Documents/matcha/data/peeter_yksiklaused_txt"

In [3]:
def sentences_to_txt_files(file, output_folder):
    with open(file, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            if line.startswith('eki_et'):
                line = re.sub('\t([0-9:]+)?\t\t', '\tx\t', line)
                parts = line.strip().split('\tx\t')
                try:
                    assert len(parts) == 2
                except AssertionError:
                    print(file, parts)
                    continue
                wav_file_name = parts[0]
                sentence = parts[1]

                txt_file_name = wav_file_name.split('.wav')[0] + '.txt'
                with open(output_folder + f'/{txt_file_name}', 'w', encoding='utf-8') as fout:
                    fout.write(sentence)

sentences_to_txt_files(sentences_1, yksiklaused_txt_path)
sentences_to_txt_files(sentences_2, yksiklaused_txt_path)

2. Üksiklausete failide ümbernimetamine

In [ ]:
# Juba failideks eraldatud tekstilised laused.
yksiklaused_txt_path = "/home/kiisu/Documents/matcha/data/peeter_yksiklaused_txt"

# Kaust, kus on juba resample'tud failid sees, võib sisaldada ka teisi, mitte-wav faile.
yksiklaused_wav_path = "/home/kiisu/Documents/matcha/data/peeter_yksiklaused_resampled_trimmed" 

In [9]:
for filename in os.listdir(yksiklaused_wav_path):
    if filename.endswith('wav') and not filename.startswith('yksiklause'):
        # 'eki_et_ptr_10015.wav'
        orig_fname = filename.split('.')[0]
        no = orig_fname.split('_')[-1]
        new_name = 'yksiklause_' + no
        shutil.copyfile(os.path.join(yksiklaused_wav_path, filename), os.path.join(yksiklaused_wav_path, f'{new_name}.wav'))

        # Üksiklausete tekstifailide ümbernimetamine
        try:
            txt_filename = orig_fname + '.txt'
            shutil.copyfile(os.path.join(yksiklaused_txt_path, txt_filename), os.path.join(yksiklaused_txt_path, f'{new_name}.txt'))
        except FileNotFoundError as e:
            print(e)

[Errno 2] No such file or directory: '/home/kiisu/Documents/matcha/data/peeter_yksiklaused_txt/eki_et_ptr_10829.txt'


3. Liiga väikeste ja suurte failide eemaldamine

In [2]:
# Juba failideks eraldatud tekstilised laused.
yksiklaused_txt_path = "/home/sandra/Documents/matcha/data/peeter_yksiklaused_txt"

# Kaust, kus on juba resample'tud failid sees, võib sisaldada ka teisi, mitte-wav faile.
yksiklaused_wav_path = "/home/sandra/Documents/matcha/data/peeter_yksiklaused_resampled_trimmed" 

In [3]:
sample_file_selection = [] # Files from the folder that are in certain size range
files_discarded = 0
files_included = 0

for file in os.listdir(yksiklaused_wav_path):
    fsize = os.path.getsize(os.path.join(yksiklaused_wav_path, file))
    if fsize < 44000 or fsize > 1000000:
        files_discarded += 1 
        continue
    files_included += 1
    sample_file_selection.append(file)

print(f'{files_included} files included in datasets. {files_discarded} files discarded.')

1694 files included in datasets. 154 files discarded.


4. Üksiklausete jagamine treening- ja validatsiooniandmeteks

In [4]:
train, val = train_test_split(sample_file_selection, train_size=0.98, shuffle=True, random_state=5)

In [5]:
# Kontrolliks
train[:10], val[:10], len(train), len(val)

(['yksiklause_00525.wav',
  'yksiklause_10349.wav',
  'yksiklause_00458.wav',
  'yksiklause_10684.wav',
  'yksiklause_10282.wav',
  'yksiklause_00700.wav',
  'yksiklause_10700.wav',
  'yksiklause_00911.wav',
  'yksiklause_10238.wav',
  'yksiklause_00652.wav'],
 ['yksiklause_10408.wav',
  'yksiklause_10464.wav',
  'yksiklause_00755.wav',
  'yksiklause_00737.wav',
  'yksiklause_10470.wav',
  'yksiklause_00689.wav',
  'yksiklause_00151.wav',
  'yksiklause_00842.wav',
  'yksiklause_10294.wav',
  'yksiklause_00997.wav'],
 1660,
 34)

5. Üksiklauste lisamine treening- ja validatsioonihulka
Muu hulgas suurendatakse validatsioonihulka testhulga arvelt (võrreldes ilma üksiklauseteta eksperimendiga).

In [6]:
def read_sentence_from_file(sentence: str, txt_folder):
    if not sentence.endswith('.txt'):
        sentence = sentence + '.txt'
    with open(os.path.join(txt_folder, sentence)) as f:
        sentence_text = f.read() + '\n'
    return sentence_text

def compose_txt_for_training(wav_file_subset, wav_file_reference_path, txt_folder):
    lines = []
    for fname in wav_file_subset:
        sentence_id = fname.split('.')[0]
        wav_reference = f'{wav_file_reference_path}/{fname}'
        sentence = read_sentence_from_file(sentence_id, txt_folder)
        lines.append(f'{wav_reference}|{sentence}')
    
    return lines

In [ ]:
# Treeningandmetesse uute andmete lisamine
import random

yksiklaused_txt_path = "/home/sandra/Documents/matcha/data/peeter_yksiklaused_txt"
yksiklaused_wav_path = "/home/sandra/Documents/matcha/data/peeter_yksiklaused_resampled_trimmed"

# Kui varasemas sisendandmestikus on vaja ka eelnevate andmete failiteed muuta (kui ei, lisa üks ja sama tee mõlemasse muutujasse)
new_path_for_previous_corpora = "/home/sandra/Documents/matcha/data/peeter_wav_resampled/"
old_path_to_replace = "/home/sandra/projects/matcha/data/peeter/peeter_wav_resampled/"

# Eelmise eksperimendi treeningandmestiku fail
train_filelist_fname = '/home/sandra/Documents/matcha/peeter_training_sets/exp-2/est_audio_text_train_filelist_4400_1000000.txt'

# Uue eksperimendi treeningandmestiku fail
new_train_path = '/home/sandra/Documents/matcha/peeter_training_sets/exp-3.2/est_audio_text_train_filelist_44000_1000000.txt'

with open(train_filelist_fname, 'r', encoding='utf-8') as f:
    old_lines = f.readlines()
    lines = [line.replace(old_path_to_replace, new_path_for_previous_corpora) for line in old_lines]

    train_set = compose_txt_for_training(train, yksiklaused_wav_path, yksiklaused_txt_path)
    train_set = lines + train_set
    random.shuffle(train_set)
    with open(new_train_path, 'w', encoding='utf-8') as fout:
        fout.writelines(train_set)

In [ ]:
# Validatsiooniandmetesse uute andmete lisamine

# Eelmise eksperimendi validatsiooni ja testandmestiku failid
validation_filelist_fname = '/home/sandra/Documents/matcha/peeter_training_sets/exp-2/est_audio_text_val_filelist_4400_1000000.txt'
test_filelist_fname = '/home/sandra/Documents/matcha/peeter_training_sets/exp-2/est_audio_text_test_filelist_4400_10000000.txt'

# Uue, üksiklausetega täiendatud eksperimendi validatsiooni ja testandmestiku failid
new_validation_path = '/home/sandra/Documents/matcha/peeter_training_sets/exp-3.2/est_audio_text_validation_filelist_44000_1000000.txt'
new_test_path = '/home/sandra/Documents/matcha/peeter_training_sets/exp-3.2/est_audio_text_test_filelist_44000_1000000.txt'


with open(validation_filelist_fname, 'r', encoding='utf-8') as f:
    old_lines = f.readlines()
    lines = [line.replace(old_path_to_replace, new_path_for_previous_corpora) for line in old_lines]
    print(f'Validatsioonihulga suurus enne: {len(lines)}')

    # Üksiklausete validatsioonihulga sisendiks sobivaks töötlemine
    val_set = compose_txt_for_training(val, yksiklaused_wav_path, yksiklaused_txt_path)

    # 1. Lisa üksiklausete validatsiooniread
    val_set = lines + val_set

    # 2. Testandmestikust andmete saamine ja testandmestiku modifitseerimine
    with open(test_filelist_fname) as ftest:
        old_test_lines = ftest.readlines()
        test_lines = [line.replace(old_path_to_replace, new_path_for_previous_corpora) for line in old_test_lines]
        
        # Validatsioonihulga jaoks uute ridade eraldamine testandmestiku arvelt
        for_val, test = train_test_split(test_lines, train_size=0.7, shuffle=True, random_state=5)
        val_set += for_val

        # Uus testfail
        with open(new_test_path, 'w', encoding='utf-8') as fout:
            print(f'Testhulga suurus: {test}')
            fout.writelines(test)
        

    # Juhuslikusta
    print(f'Validatsioonihulga suurus: {len(val_set)}')
    random.shuffle(val_set)
    with open(new_validation_path, 'w', encoding='utf-8') as fout:
        fout.writelines(val_set)


Validatsioonihulga suurus enne: 49
Testhulga suurus: ['/home/sandra/Documents/matcha/data/peeter_wav_resampled/lause01190.wav|Nõnda jäigi Krõõda suu nagu naeratama, ilma et oleks küsinud, kas on sündinud tütar või poeg.\n', '/home/sandra/Documents/matcha/data/peeter_wav_resampled/lause02578.wav|„Oli uimastav balliöö” ja Arsis Dikk ja Doffi\n', '/home/sandra/Documents/matcha/data/peeter_wav_resampled/lause01012.wav|Kotka Karla nagu ei pannud leivaandja sõnu tähelegi.\n', '/home/sandra/Documents/matcha/data/peeter_wav_resampled/lause00234.wav|Ta oli endiselt elegantne.\n', '/home/sandra/Documents/matcha/data/peeter_wav_resampled/lause04868.wav|„Hiline pärastlõuna sobib.”\n', '/home/sandra/Documents/matcha/data/peeter_wav_resampled/lause02860.wav|Aleksi hääl oli viimase küsimuse ajal olnud veel endiselt mahe.\n', '/home/sandra/Documents/matcha/data/peeter_wav_resampled/lause05151.wav|Kui nad ka mingid õppinud mehed pole, siis on nad ometi ausamad kui mõni teine siin saalis, kes praalib RO